# 25장 · 프롬프트 체이닝 · 앞 호출의 출력을 다음 입력으로 전달하기

프롬프트 체이닝은 첫 번째 모델 호출의 출력을 두 번째 호출의 입력으로 넘기는 기법입니다. 코드에서 중간값을 변수에 담아 두면 다음 모델이 실제로 무엇을 읽는지 확인할 수 있습니다.

## 두 번의 ask를 연결하기

체이닝은 앞 호출의 반환값을 다음 호출의 입력으로 쓰는 방법입니다. 아래에서 ‘사과’를 번역한 단어가 다음 예문에 쓰이는지 두 줄의 출력을 비교합니다.

`ask(지시, 입력)`은 OpenAI SDK로 요청하고 답을 문자열로 돌려주는 수업용 함수입니다. 수업 커널에서 먼저 아래 셀로 함수를 정의하세요. 새 커널에서 openai를 찾지 못하면 `%pip install openai`를 실행하세요.

이 장의 `ask(instruction, material)`는 교재에서 정의합니다. 내부의 `client.responses.create`가 SDK 메서드입니다. 두 인자는 각각 모델의 작업 지시와 읽을 자료가 되고, `return response.output_text`로 호출 정보 중 답 문자열만 다음 단계에 넘깁니다. `timeout=45`는 요청 제한 시간을 초 단위로 설정하며 `max_retries=1`은 재시도 가능한 전송 오류의 추가 시도 횟수입니다. 답을 다시 쓰는 횟수와는 별개입니다.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
word = ask("영어 단어 하나로 번역하세요.", "사과")
sentence = ask("이 단어로 짧은 영어 예문을 쓰세요.", word)
print(word)
print(sentence)

첫 줄은 첫 호출이 돌려준 단어이고 둘째 줄은 그 단어로 만든 예문입니다. 둘째 호출에 넘긴 word의 값이 첫 줄에 나온 단어입니다.

## 응용 실습 · 두 단계로 문의에 답하기

### 문의 원문과 두 응답을 같은 파일에 남기기

대여 문의를 받은 담당자가 초안에서 날짜를 빠뜨렸습니다. 규정과 회원의 문의를 준비했습니다. 메모와 답을 같은 파일에 담으면 다음 시간에 실제 초안을 다시 열어 누락된 조건을 찾아볼 수 있습니다. 수업 묶음의 압축을 모두 풀고 노트북을 여세요.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

## 두 번째 호출에 필요한 정보를 먼저 정하기

체인에서는 첫 호출의 반환값이 다음 호출의 입력이 됩니다. 두 번째 호출에 원래 질문을 다시 주지 않는다면, 필요한 날짜와 두 개의 물음이 첫 메모에 남아야 합니다. 실습 자료의 Q1에서 첫 호출이 보존해야 할 정보를 표시하세요.

## 응용에 사용할 자료 열기

실습 묶음의 압축을 풀고 노트북과 `pre-mid` 폴더를 나란히 두세요. 다음 셀은 대여 규정 파일을 읽어 `rules`에 저장합니다. 응용에서도 앞에서 정의한 `ask`를 사용합니다.

In [ ]:
from pathlib import Path

rules = (next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/sources/loan-rules.md").is_file()) / "pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")

In [ ]:
question = '6월 10일에 A형을 빌리기로 했고 승인 댓글도 받았어요. 수업이 늦게 끝나는데 친구가 대신 받아도 될까요? 제가 직접 받으면 케이블은 언제 어디로 돌려주면 되나요?'
print(rules)

R2는 수령 조건, R3은 반납 기한과 물건을 정합니다. R3의 “다음 날”을 실제 날짜로 쓰려면 수령 날짜가 남아 있어야 합니다. 모델이 어떤 답을 써야 할지 판단할 때 원문에서 보존할 사실과 해결할 요청을 나누어 보세요.

> **R2**　승인받은 회원은 대여일 오후 3시부터 5시까지 가온관 1층 안내 데스크에서 기기를 받습니다. 학생증을 보여 주세요. 친구가 대신 받을 수 없습니다.

> **R3**　수령 다음 날 오후 5시까지 가온관 1층 안내 데스크로 반납해 주세요. 기기와 함께 빌린 케이블도 가져오세요.

## 한 번에 답해 기준 답변 남기기

한 번 호출해 얻은 답을 기준으로 삼습니다. 두 단계로 나눴다는 이유만으로 더 나은 답이 되지는 않습니다. 첫 답도 이미 충분히 정확할 수 있습니다.

In [ ]:
direct = ask("대여 담당자로서 문의에 답하세요. 규정에서 근거가 되는 조항을 붙이세요.", rules + "\n회원 문의\n" + question)
print(direct)

직접 받은 답 옆에 대리 수령 가능 여부, 기기와 케이블, 반납 기한, 반납 장소를 적습니다. 날짜가 상대 표현이라면 언제부터 계산하는지도 읽으세요. 문장 길이는 이 네 조건의 정확성을 알려 주지 않습니다.

## 두 번째 ask 호출에 첫 응답 넘기기

워크플로는 어떤 일을 어느 순서로 수행하고 결과를 누구에게 넘길지 정한 절차입니다. 여기서는 첫 번째 호출이 인수인계 메모를 쓰고 두 번째 호출이 그 메모와 규정을 받아 답합니다. 두 번째 호출에 첫 원문을 자동으로 전달하는 기능은 없습니다. 무엇을 넘길지 우리가 고릅니다.

프롬프트 체이닝은 한 호출의 결과를 다음 호출의 입력에 넣는 방법입니다. 첫 단계 `ask(take_notes, question)`은 문자열 하나를 반환해 `notes`에 담습니다. 다음 모델이 앞선 호출을 기억한다고 가정할 수 없으므로 `second_input`을 직접 조립합니다. 첫 메모에서 “친구가 대신”만 지우면 두 번째 모델에 어떤 질문이 전달되지 않는지 짚어 보세요.

In [ ]:
take_notes = "회원이 말한 사실과 해결할 요청을 나누세요. 기종·날짜·승인 여부와 요청을 빠짐없이 남기세요. 답은 아직 쓰지 마세요."
notes = ask(take_notes, question)
print(notes)

`notes`에는 첫 호출이 반환한 글이 들어 있습니다. 다음 셀은 `rules`와 `notes`를 이어서 `second_input`을 만듭니다. 답에서 물음 하나가 빠졌다면 `print(second_input)`으로 그 물음이 두 번째 호출까지 전달됐는지 확인하세요.

In [ ]:
write_answer = "규정과 인수인계 메모로 회원에게 짧게 답하세요. 메모의 요청을 각각 처리하고 근거 조항을 붙이세요."
# 두 번째 호출에 보낼 내용을 보관해 문제가 생기면 다시 읽습니다.
second_input = rules + "\n인수인계 메모\n" + notes
draft = ask(write_answer, second_input)
print(draft)

기한이 틀렸다면 먼저 메모에 날짜가 남았는지 확인하세요. 날짜가 있었다면 규정 해석 또는 답안 작성에서 어긋난 것입니다. 단계를 나누면 앞 단계에서 이미 사라진 사실을 찾을 수 있습니다.

## 중간 변수만 바꿔 답이 어긋난 단계 찾기

두 번째 호출에 넘길 `notes`만 짧은 시험 입력 `missing_note`로 바꿉니다. 지시와 규정은 그대로입니다. 반납에 관한 말이 빠졌을 때 아래 줄에서 모델이 받은 입력과 내놓은 답을 대조해 보세요.

In [ ]:
missing_note = "6월 10일 A형 승인 완료. 친구가 대신 받을 수 있는지 질문함."
print(ask(write_answer, rules + "\n인수인계 메모\n" + missing_note))

반납 기한이나 케이블이 빠졌다면 첫 단계의 누락이 이후 단계에 영향을 준 사례입니다. 반대로 모델이 반납까지 안내했더라도 누락된 질문을 메모에서 되살린 것은 아닙니다. 다른 문의에서도 같은 동작을 보장할 근거가 있는지 따져 보세요.

## 두 단계가 맞물린 메모와 답을 읽어 보기

**집필자가 쓴 메모 예시**　확인한 사실은 6월 10일 A형 대여 승인입니다. 요청 하나는 친구의 대리 수령 가능 여부, 다른 하나는 케이블 반납 시각과 장소입니다.

**이 메모를 받은 담당자의 답 예시**　친구가 대신 받을 수 없습니다. 6월 10일 오후 3~5시에 학생증을 지참해 가온관 1층 안내 데스크로 오세요. 기기와 케이블은 다음 날인 11일 오후 5시까지 같은 데스크에 반납하세요.

날짜 계산은 메모의 10일과 R3의 다음 날에서 나옵니다. 수령 시간과 장소는 R2에서 나옵니다. 자신의 실행 결과에서 각 조건이 어디에서 왔는지 짚어 보세요.

## 모델에게 일을 넘기기 전에 써 보는 인수인계 메모

두 명이 원문을 2분 동안 읽습니다. 첫 사람은 원문을 덮은 뒤 “이 회원은 무엇을 이미 마쳤나? 담당자가 답해야 할 것은 무엇인가?”를 한 줄씩 씁니다. 둘째 사람은 메모와 R1~R4만 보고 답합니다. 두 사람이 다시 원문을 열어 비교하세요. 학생증이 필요하다는 말이 빠졌는지와 반납 문의 자체가 빠졌는지는 다른 문제입니다. 전자는 규정에서 찾을 수 있지만 후자는 메모만으로 복원하기 어렵습니다. 이 차이가 첫 호출의 지시를 결정합니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 처음 보는 문의를 인수인계하기

개인 실습 90분. 자료의 Q4를 읽고, 첫 호출 전에 보존할 날짜와 두 물음을 종이에 씁니다. 기준 답변, 요구 추출, 초안 작성을 직접 실행하세요. 원문→메모→답에서 처음 사라진 사실 하나를 찾아 첫 지시만 고칩니다. 누락이 없다면 “댓글만 쓰면 되는지”를 의도적으로 뺀 메모로 다시 시험하세요. 짝은 어떤 입력이 바뀌었는지 기록만 보고 찾아야 합니다.

> **R4**　대여 기간은 한 번, 하루까지 연장할 수 있습니다. 원래 반납일 정오까지 신청 게시글에 연장 요청 댓글을 남겨 주세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다.

In [ ]:
new_question = '6월 10일에 A형을 직접 받았어요. 반납을 하루 미루려면 언제까지 어디에 남기면 되나요? 댓글만 쓰면 연장이 끝나는지도 궁금해요.'
my_case = None
# 이 문의로 ask를 두 번 호출하고 메모와 답을 얻으세요.
# my_case에 question, notes, draft 세 값을 담으면 다음 장에서 내 결과를 읽습니다.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>new_question = &#x27;6월 10일에 A형을 직접 받았어요. 반납을 하루 미루려면 언제까지 어디에 남기면 되나요? 댓글만 쓰면 연장이 끝나는지도 궁금해요.&#x27;
direct_again = ask(&quot;회원의 문의에 규정으로 답하고 근거를 붙이세요.&quot;, rules + &quot;\n&quot; + new_question)
print(&quot;한 번의 답&quot;, direct_again)
for instruction in (take_notes, take_notes + &quot; 연장 요청 댓글을 쓰는 일과 승인이 끝나는 조건을 별도 질문으로 보존하세요.&quot;):
    memo = ask(instruction, new_question)
    received = rules + &quot;\n메모\n&quot; + memo
    print(&quot;메모&quot;, memo)
    answer_for_member = ask(write_answer, received)
    print(&quot;메모로 작성한 답&quot;, answer_for_member)
my_case = {&quot;question&quot;: new_question, &quot;notes&quot;: memo, &quot;draft&quot;: answer_for_member}</code></pre>
<p>대조 예시. 6월 10일 수령 → 원래 반납일은 11일. 원래 반납일 정오까지 원래 게시글에 요청하고 담당자 승인 댓글을 받아야 합니다. 표현이 달라도 네 조건의 근거를 R3·R4에서 제시할 수 있으면 됩니다.</p>
</details>

## 초안과 작성 근거 보관하기

실습에서 만든 my_case가 있으면 그 문의를 씁니다. 아직 작성하지 않았으면 위에서 실행한 수령·반납 예제를 보관합니다. 26장에서 같은 답을 검토해 달라고 요청할 수 있게 원문, 첫 호출의 메모, 둘째 호출의 초안을 저장하세요. 파일을 다시 열었을 때 원문과 초안이 각각 같은지 확인합니다.

In [ ]:
record = my_case if my_case is not None else {"question": question, "notes": notes, "draft": draft}
save_record(root, 25, record)
actual = read_previous(root, 25)
assert actual["question"] == record["question"] and actual["draft"] == record["draft"]
print("원문과 초안 일치", True)

작성 메모와 초안이 같은 문의에 묶였는지 봅니다. 다음 셀은 저장한 파일을 다시 읽어 원문과 답이 보존됐는지 검사합니다. 실시간 모델을 실행하지 않았으면 아래 셀만 골라 실행해 교재 예시의 자료 구조를 읽을 수 있습니다. 첫 줄이 파일의 출처입니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

saved = read_previous(root, 25)
print("문의와 초안이 모두 있나요?", bool(saved["question"] and saved["draft"]))

교재 복귀 예시 · chapter25.json
문의와 초안이 모두 있나요? True


## 다음 작업에 가져갈 기록

기준 답변과 중간 메모, 수정한 지시를 저장하세요. 다음 장에서는 정보가 빠진 신청까지 이 작성 절차에 넣어도 되는지 결정합니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)